In [0]:
from datetime import datetime
import urllib.request
import tempfile
import os

from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS workspace.pb_threat_intelligence;

CREATE VOLUME IF NOT EXISTS workspace.pb_threat_intelligence.feed_snapshots;

In [0]:
feed_a_link = 'https://raw.githubusercontent.com/StevenBlack/hosts/master/hosts'
feed_b_link = 'https://malware-filter.gitlab.io/malware-filter/urlhaus-filter-agh.txt'
feed_c_link = 'https://badmojr.gitlab.io/1hosts/Lite/adblock.txt'
feed_d_link = 'https://raw.githubusercontent.com/jdlingyu/ad-wars/master/hosts'

feed_links = {
    'feed_a': feed_a_link,
    'feed_b': feed_b_link,
    'feed_c': feed_c_link,
    'feed_d': feed_d_link,
}

volume_dir = f'/Volumes/workspace/pb_threat_intelligence/feed_snapshots'

def download_feed(name, url):
    """Download a feed to a temporary local file and return the path."""
    tmp_path = f"{volume_dir}/{name}.txt"
    urllib.request.urlretrieve(url, tmp_path)
    print(f"Downloaded {name} ({url}) -> {tmp_path}")
    return tmp_path

# Download all feeds
downloaded = {name: download_feed(name, url) for name, url in feed_links.items()}

schema = StructType([StructField("raw_line", StringType(), True)])

feed_dfs = {}
for name, path in downloaded.items():
    df = (spark.read
          .schema(schema)
          .text(path)
          .withColumnRenamed("value", "raw_line")
          .withColumn("feed_name", F.lit(name)))
    feed_dfs[name] = df
    print(f"{name}: {df.count()} rows loaded")

# all_feeds_df = functools.reduce(lambda a, b: a.unionByName(b), feed_dfs.values())
# all_feeds_df.display()

Downloaded feed_a (https://raw.githubusercontent.com/StevenBlack/hosts/master/hosts) -> /Volumes/workspace/pb_threat_intelligence/feed_snapshots/07102026/feed_a.txt
Downloaded feed_b (https://malware-filter.gitlab.io/malware-filter/urlhaus-filter-agh.txt) -> /Volumes/workspace/pb_threat_intelligence/feed_snapshots/07102026/feed_b.txt
Downloaded feed_c (https://badmojr.gitlab.io/1hosts/Lite/adblock.txt) -> /Volumes/workspace/pb_threat_intelligence/feed_snapshots/07102026/feed_c.txt
Downloaded feed_d (https://raw.githubusercontent.com/jdlingyu/ad-wars/master/hosts) -> /Volumes/workspace/pb_threat_intelligence/feed_snapshots/07102026/feed_d.txt
feed_a: 79056 rows loaded
feed_b: 18101 rows loaded
feed_c: 89840 rows loaded
feed_d: 1738 rows loaded


In [0]:
feed_dfs['feed_a'].display()

raw_line,feed_name
# Title: StevenBlack/hosts,feed_a
#,feed_a
"# This hosts file is a merged collection of hosts from reputable sources,",feed_a
# with a dash of crowd sourcing via GitHub,feed_a
#,feed_a
# Date: 02 October 2026 19:58:39 (UTC),feed_a
"# Number of unique domains: 72,233",feed_a
#,feed_a
# Fetch the latest version of this file: https://raw.githubusercontent.com/StevenBlack/hosts/master/hosts,feed_a
# Project home page: https://github.com/StevenBlack/hosts,feed_a


In [0]:
feed_dfs['feed_a'].tail(1000)

[Row(raw_line='0.0.0.0 mytop-in.net', feed_name='feed_a'),
 Row(raw_line='0.0.0.0 n69.com', feed_name='feed_a'),
 Row(raw_line='0.0.0.0 naj.sk', feed_name='feed_a'),
 Row(raw_line='0.0.0.0 nakedly.ai', feed_name='feed_a'),
 Row(raw_line='0.0.0.0 nappyneck.com', feed_name='feed_a'),
 Row(raw_line='0.0.0.0 nastydollars.com', feed_name='feed_a'),
 Row(raw_line='0.0.0.0 nativeroll.tv', feed_name='feed_a'),
 Row(raw_line='0.0.0.0 navegg.com', feed_name='feed_a'),
 Row(raw_line='0.0.0.0 navigator.io', feed_name='feed_a'),
 Row(raw_line='0.0.0.0 navrcholu.cz', feed_name='feed_a'),
 Row(raw_line='0.0.0.0 nebulacrescent.com', feed_name='feed_a'),
 Row(raw_line='0.0.0.0 nedstatbasic.net', feed_name='feed_a'),
 Row(raw_line='0.0.0.0 needyneedle.com', feed_name='feed_a'),
 Row(raw_line='0.0.0.0 neighborlywatch.com', feed_name='feed_a'),
 Row(raw_line='0.0.0.0 neocounter.neoworx-blog-tools.net', feed_name='feed_a'),
 Row(raw_line='0.0.0.0 net-filter.com', feed_name='feed_a'),
 Row(raw_line='0.0.0.0

In [0]:
system_hosts = [
    "localhost", "localhost.localdomain", "local", "broadcasthost",
    "ip6-localhost", "ip6-loopback", "ip6-localnet",
    "ip6-mcastprefix", "ip6-allnodes", "ip6-allrouters", "ip6-allhosts",
    "0.0.0.0"
]

In [0]:
domains = (
    feed_dfs['feed_a']
    #.withColumn("content", F.trim(F.regexp_replace("value", r"#.*$", "")))
    .filter(F.col("raw_line").rlike(r"^0\.0\.0\.0\s+"))
    .withColumn("tokens", F.split("raw_line", r"\s+"))
    .withColumn(
        "domain",
        F.explode(F.expr("slice(tokens, 2, size(tokens) - 1)"))
    )
    .withColumn("domain", F.lower("domain"))
    .filter(~F.col("domain").isin(system_hosts))
    .select("domain")
)

In [0]:
domains.count() # 72,878 vs Based on file "Number of unique domains: 72,233"

72878

In [0]:
domains.display()

domain
ad-assets.futurecdn.net
ck.getcookiestxt.com
eu1.clevertap-prod.com
wizhumpgyros.com
coccyxwickimp.com
webmail-who-int.000webhostapp.com
010sec.com
01mspmd5yalky8.com
0byv9mgbn0.com
ns6.0pendns.org


In [0]:
domains.filter(F.col("domain").startswith("#")).display()

domain
#
#
#
#
#
#
#ad
#[ban
#[tracking.cookie]
#[webbug]


In [0]:
domains.filter(~F.col("domain").startswith("#")).count(), domains.filter(~F.col("domain").startswith("#")).dropDuplicates().count()

(72520, 53671)

In [0]:
domains.filter(F.trim(F.col("domain")) == '').count(), domains.filter(F.col("domain").isNull()).count()

(0, 0)

In [0]:
domains.withColumn("domain", F.regexp_replace("domain", r"^www\.", "")).dropDuplicates().count()

# so it deduplication after getting rid of "www."" wasn't calculated in source count

53782

In [0]:
from pyspark.sql.window import Window

feed_dfs['feed_a'].withColumn('row_number', F.row_number().over(Window.orderBy(F.lit(1)))).filter(F.col('raw_line').startswith('#')).display()

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


raw_line,feed_name,row_number
# Title: StevenBlack/hosts,feed_a,1
#,feed_a,2
"# This hosts file is a merged collection of hosts from reputable sources,",feed_a,3
# with a dash of crowd sourcing via GitHub,feed_a,4
#,feed_a,5
# Date: 02 October 2026 19:58:39 (UTC),feed_a,6
"# Number of unique domains: 72,233",feed_a,7
#,feed_a,8
# Fetch the latest version of this file: https://raw.githubusercontent.com/StevenBlack/hosts/master/hosts,feed_a,9
# Project home page: https://github.com/StevenBlack/hosts,feed_a,10


In [0]:
feed_dfs['feed_a'].filter(
    ~((F.col('raw_line').startswith('#')) | (F.col('raw_line').startswith('0.0.0.0')) | (F.col('raw_line') == ''))
).display()

raw_line,feed_name
127.0.0.1 localhost,feed_a
127.0.0.1 localhost.localdomain,feed_a
127.0.0.1 local,feed_a
255.255.255.255 broadcasthost,feed_a
::1 localhost,feed_a
::1 ip6-localhost,feed_a
::1 ip6-loopback,feed_a
fe80::1%lo0 localhost,feed_a
ff00::0 ip6-localnet,feed_a
ff00::0 ip6-mcastprefix,feed_a


In [0]:
# the only malware category
feed_dfs['feed_b'].display()

# 12 hours (update frequency) - will need to be update 2 times per day in 12 hours intervals
# last update was on 2026-10-06 12:07:27 so assuming we can't proceed with event-based trigger
# the trigger should be arround 12:10 - 12:15 (AM/PM) with retry option when the source file was not updated yet

raw_line,feed_name
! Title: Malicious URL Blocklist (AdGuard Home),feed_b
! Updated: 2026-10-06T12:07:27Z,feed_b
! Expires: 12 hours (update frequency),feed_b
! Homepage: https://gitlab.com/malware-filter/urlhaus-filter,feed_b
! License: https://gitlab.com/malware-filter/urlhaus-filter#license,feed_b
! Source: https://urlhaus.abuse.ch/api/,feed_b
||0.232.205.92.host.secureserver.net^,feed_b
||01ejjpa2.behtarin-site-shartbandi.com^,feed_b
||01pr.com.tr^,feed_b
||02teste.cc^,feed_b


In [0]:
feed_dfs['feed_b'].filter(
    ~((F.col('raw_line').startswith('!')) | (F.col('raw_line').startswith('||')))
).display()

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


raw_line,feed_name,row_number


In [0]:
feed_dfs['feed_c'].display()

raw_line,feed_name
! Title: 1Hosts (Lite),feed_c
! Last modified: 2024-05-12T06:39:45.794Z,feed_c
! ..............,feed_c
! |URL: https://github.com/badmojr/1Hosts,feed_c
"! |Count: 89,822 rules!",feed_c
! |Size: 1.95 MB,feed_c
! |,feed_c
! |Credits & ©: https://o0.pages.dev/-data/lists/assets.txt,feed_c
"! |Licence: MPLv2,",feed_c
! | https://www.mozilla.org/en-US/MPL/2.0/,feed_c


In [0]:
feed_dfs['feed_b'].filter(
    ~((F.col('raw_line').startswith('!')) | (F.col('raw_line').startswith('||')))
).display()

raw_line,feed_name


In [0]:
feed_dfs['feed_d'].display()

raw_line,feed_name
127.0.0.1 localhost,feed_d
::1 localhost,feed_d
,feed_d
#pkg=white.ad.web.com;【提示误伤】选择性不√以下子项,feed_d
#------------------------------------------,feed_d
127.0.0.1 livew.l.qq.com #-> 腾讯视频,feed_d
127.0.0.1 t7z.cupid.iqiyi.com #-> 爱奇艺,feed_d
127.0.0.1 wxsnsdy.wxs.qq.com #-> 微信,feed_d
#------------------------------------------,feed_d
#--下列会影响奖励形式的广告(小程序\小游戏等),feed_d
